# EyeAI AMD3 — Run 03 Fixed Unfreeze + Center/Macula Crop

This notebook runs the third improvement experiment:

- EfficientNetV2-S 384
- Fixed tail unfreeze by parameter budget
- Center/macula crop with scale 0.65
- Raw best checkpoint saving separated from early-stopping `min_delta`
- Hugging Face token setup and robust model pre-download

Before running:

- Kaggle Internet: ON
- Accelerator: GPU
- Secrets: `GITHUB_TOKEN`, `HF_TOKEN`
- Add HYAMD dataset input.


In [ ]:
# ============================================================
# Cell 1 - Global settings
# ============================================================

from pathlib import Path
import os
import subprocess
import json
import shutil

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"

REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")
OUTPUT_DIR = Path("/kaggle/working/eyeai_binary_ensemble")

RUN03_CONFIG = "configs/train_efficientnetv2_binary_macula_fixed_unfreeze.yaml"

print("REPO_DIR:", REPO_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RUN03_CONFIG:", RUN03_CONFIG)


In [ ]:
# ============================================================
# Cell 2 - Clone / pull GitHub repository
# ============================================================

from kaggle_secrets import UserSecretsClient
from pathlib import Path
import subprocess
import os
import shutil

user_secrets = UserSecretsClient()
github_token = user_secrets.get_secret("GITHUB_TOKEN")

if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print("Repository exists. Pulling latest changes...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    if REPO_DIR.exists():
        print("REPO_DIR exists but is not a git repo. Removing it...")
        shutil.rmtree(REPO_DIR)
    print("Cloning repository...")
    subprocess.run(["git", "clone", "-b", BRANCH, repo_url, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

print("\nCurrent directory:", Path.cwd())
print("\nTop-level files:")
for p in sorted(REPO_DIR.iterdir()):
    print("-", p.name)


In [ ]:
# ============================================================
# Cell 3 - Verify repository structure and Run 03 files
# ============================================================

from pathlib import Path
import os

os.chdir(REPO_DIR)

required_files = [
    "requirements.txt",
    "pyproject.toml",
    RUN03_CONFIG,
    "scripts/prepare_hyamd.py",
    "scripts/train_binary.py",
    "src/eyeai/data/transforms.py",
    "src/eyeai/training/train_binary.py",
    "src/eyeai/models/cnn_models.py",
]

missing = []

for file_path in required_files:
    exists = (REPO_DIR / file_path).exists()
    print(f"{file_path}: {exists}")
    if not exists:
        missing.append(file_path)

if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print("\nRepository structure is OK.")


In [ ]:
# ============================================================
# Cell 4 - Install requirements and EyeAI package
# ============================================================

import subprocess
import os
from pathlib import Path

os.chdir(REPO_DIR)

print("Installing requirements...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Installing EyeAI package in editable mode...")
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

print("EyeAI package installed correctly.")


In [ ]:
# ============================================================
# Cell 5 - Hugging Face authentication
# ============================================================

from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

if not hf_token:
    raise RuntimeError("HF_TOKEN secret is missing.")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
os.environ["HF_HUB_TOKEN"] = hf_token

print("HF_TOKEN loaded:", bool(os.environ.get("HF_TOKEN")))

try:
    from huggingface_hub import login, HfApi
    login(token=hf_token, add_to_git_credential=False)
    who = HfApi().whoami(token=hf_token)
    print("Hugging Face authentication OK.")
    print("HF user:", who.get("name", "unknown"))
except Exception as e:
    print("Hugging Face login check failed:")
    print(repr(e))


In [ ]:
# ============================================================
# Cell 6 - Clean incomplete Hugging Face downloads and locks
# ============================================================

from pathlib import Path

cache_dirs = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path("/kaggle/working/.cache/huggingface/hub"),
]

removed = 0

for cache_dir in cache_dirs:
    if not cache_dir.exists():
        continue
    print("Checking cache:", cache_dir)
    for pattern in ["*.incomplete", "*.lock"]:
        for p in cache_dir.rglob(pattern):
            try:
                p.unlink()
                removed += 1
                print("Removed:", p)
            except Exception as e:
                print("Could not remove:", p, repr(e))

print("Removed incomplete/lock files:", removed)


In [ ]:
# ============================================================
# Cell 7A - Optional: remove broken EfficientNetV2-S HF cache
# ============================================================

from pathlib import Path
import shutil

patterns = [
    "models--timm--tf_efficientnetv2_s.in21k_ft_in1k",
]

cache_roots = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path("/kaggle/working/.cache/huggingface/hub"),
]

removed = []

for root in cache_roots:
    if not root.exists():
        continue
    for pattern in patterns:
        for p in root.glob(pattern):
            try:
                shutil.rmtree(p)
                removed.append(str(p))
                print("Removed cache folder:", p)
            except Exception as e:
                print("Could not remove:", p, repr(e))

print("Removed folders:", len(removed))


In [ ]:
# ============================================================
# Cell 7B - Pre-download only EfficientNetV2-S model.safetensors
# ============================================================

import os
from huggingface_hub import hf_hub_download

repo_id = "timm/tf_efficientnetv2_s.in21k_ft_in1k"

print("Downloading only model.safetensors from:", repo_id)

weights_path = hf_hub_download(
    repo_id=repo_id,
    filename="model.safetensors",
    token=os.environ.get("HF_TOKEN"),
    resume_download=True,
    force_download=False,
)

print("Downloaded weights:")
print(weights_path)


In [ ]:
# ============================================================
# Cell 8 - Check Kaggle input paths and config
# ============================================================

from pathlib import Path
import os
import yaml

os.chdir(REPO_DIR)

print("Current directory:", Path.cwd())

print("\nKaggle input folders:")
for p in sorted(Path("/kaggle/input").glob("*")):
    print("-", p)

config_path = REPO_DIR / RUN03_CONFIG
print("\nConfig path:", config_path)
print("Config exists:", config_path.exists())

with open(config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("\nTop-level config keys:")
print(list(cfg.keys()))

print("\nData config:")
print(cfg.get("data", {}))

hyamd_path = None

if "paths" in cfg and "hyamd_input_dir" in cfg["paths"]:
    hyamd_path = Path(cfg["paths"]["hyamd_input_dir"])
elif "data" in cfg and "hyamd_input_dir" in cfg["data"]:
    hyamd_path = Path(cfg["data"]["hyamd_input_dir"])
elif "data" in cfg and "input_dir" in cfg["data"]:
    hyamd_path = Path(cfg["data"]["input_dir"])
elif "data" in cfg and "hyamd_root" in cfg["data"]:
    hyamd_path = Path(cfg["data"]["hyamd_root"])
else:
    raise KeyError("Could not find HYAMD path in config.")

print("\nHYAMD path from config:")
print(hyamd_path)
print("Exists:", hyamd_path.exists())

if not hyamd_path.exists():
    print("\nAvailable folders under /kaggle/input:")
    for p in sorted(Path("/kaggle/input").rglob("*")):
        if p.is_dir():
            print("-", p)
    raise FileNotFoundError(f"HYAMD input path does not exist: {hyamd_path}")

print("\nHYAMD input path is OK.")


In [ ]:
# ============================================================
# Cell 9 - Prepare HYAMD data and binary splits
# ============================================================

import os
from pathlib import Path

os.chdir(REPO_DIR)

!python -u scripts/prepare_hyamd.py --config configs/train_efficientnetv2_binary_macula_fixed_unfreeze.yaml


In [ ]:
# ============================================================
# Cell 10 - Inspect generated splits
# ============================================================

from pathlib import Path
import pandas as pd

split_dir = OUTPUT_DIR / "HYAMD_raw" / "splits"

print("Split directory:", split_dir)
print("Exists:", split_dir.exists())

for name in ["train.csv", "val.csv", "test.csv"]:
    p = split_dir / name
    print("\n", "=" * 80)
    print(name, "exists:", p.exists())
    if p.exists():
        df = pd.read_csv(p)
        print("shape:", df.shape)
        if "binary_label" in df.columns:
            print("binary_label distribution:")
            print(df["binary_label"].value_counts().sort_index())
        if "label" in df.columns:
            print("original label distribution:")
            print(df["label"].value_counts().sort_index())


In [ ]:
# ============================================================
# Cell 11 - Train EfficientNetV2-S Run 03
# ============================================================

import os
from pathlib import Path

os.chdir(REPO_DIR)

!python -u scripts/train_binary.py --config configs/train_efficientnetv2_binary_macula_fixed_unfreeze.yaml


In [ ]:
# ============================================================
# Cell 12 - Show generated output files
# ============================================================

from pathlib import Path

root = OUTPUT_DIR

print("Output root:", root)
print("Exists:", root.exists())

important_parts = [
    "checkpoints",
    "logs",
    "predictions",
    "thresholds",
    "ensemble",
]

for p in sorted(root.rglob("*")):
    if p.is_file() and any(part in str(p) for part in important_parts):
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"{p} | {size_mb:.2f} MB")


In [ ]:
# ============================================================
# Cell 13 - Read training summaries
# ============================================================

from pathlib import Path
import json
import pandas as pd

logs_dir = OUTPUT_DIR / "logs"
print("Logs directory:", logs_dir)

summary_files = sorted(logs_dir.glob("*macula*summary*.json"))
history_files = sorted(logs_dir.glob("*macula*history*.csv"))

print("\nSummary files:")
for p in summary_files:
    print("-", p)

print("\nHistory files:")
for p in history_files:
    print("-", p)

for p in summary_files:
    print("\n", "=" * 90)
    print("SUMMARY:", p.name)
    with open(p, "r", encoding="utf-8") as f:
        summary = json.load(f)
    print(json.dumps(summary, indent=2, ensure_ascii=False))

if history_files:
    latest_history = history_files[-1]
    print("\n", "=" * 90)
    print("Latest history:", latest_history)
    hist = pd.read_csv(latest_history)
    display(hist.tail(10))
